# Redaktionelle Abbildungen reproduzieren und Seiten prüfen

Die freigegebene Redaktionsrunde betrifft PV, Modellübersicht, thermische Hülle und Haustechnik sowie zugehörige Erläuterungen bei Varianten und Deklaration. Die Preset-Erweiterung ist nur in `docs/development/preset_catalog_proposal.md` skizziert.

Die Originalaufnahmen liegen thematisch unter `website/illustrations/pv/` und `website/illustrations/hvac/`. `scripts/website_editorial_figures.py` erzeugt daraus SVG-Ansichten mit Ausschnitt und nummerierten Rahmen. Die Screenshots bleiben unverändert; die SVG-Ebene legt nur den sichtbaren Bereich und die Markierungen fest. Die Koordinaten beziehen sich auf die Originalpixel. Nach Austausch eines Originals die Koordinaten dort anpassen und alle betroffenen Abbildungen visuell prüfen. Kein zweiter Renderer wird im Notebook gepflegt.

Die Texte erläutern die Bedienung mit den Originalfeldern und den fachlichen Angaben des Projektteams. Die Website-Screenshots testen nicht die Excel-Berechnung. Quarto muss installiert sein; bereits vorhandene Website-Ressourcen werden wiederverwendet.


In [ ]:
from pathlib import Path
import sys
import subprocess
import xml.etree.ElementTree as ET

root = Path.cwd()
if not (root / "scripts/website_editorial_figures.py").exists():
    root = root.parent
sys.path.insert(0, str(root / "scripts"))
from website_editorial_figures import build
figures = build(root)
for figure in figures:
    ET.parse(figure)
print(f"{len(figures)} SVG-Abbildungen erzeugt.")
pages = ("pv", "model", "envelope", "hvac", "variants", "declaration")
for page in pages:
    subprocess.run(["quarto", "render", f"website/{page}.qmd"], cwd=root, check=True)
print("Vorschau:", root / "website/_site")


In [ ]:
from html.parser import HTMLParser

class Images(HTMLParser):
    def __init__(self):
        super().__init__()
        self.sources = []
    def handle_starttag(self, tag, attrs):
        if tag == "img":
            self.sources.append(dict(attrs).get("src", ""))

for page in pages:
    html_path = root / f"website/_site/{page}.html"
    parser = Images()
    parser.feed(html_path.read_text(encoding="utf-8"))
    for source in parser.sources:
        if source and not source.startswith(("http:", "https:", "data:")):
            assert (html_path.parent / source).exists(), (page, source)
print("Alle lokalen Bildreferenzen der sechs Seiten vorhanden.")


## Visuelle Abnahme im Browser

Mit `python -m http.server 8766 --bind 127.0.0.1 --directory website/_site` im Repository lokal bereitstellen (oder den bereits laufenden Vorschau-Server verwenden).

1. `pv.html`: Modus, Dimensionierung und eigene Profile müssen lesbar sein; die Markierungen 1–3 entsprechen den Erläuterungen. Bildvergrößerung testen.
2. `hvac.html`: Heizungs-, Kühlungs- und Lüftungsansichten einschließlich Legenden, FreeCooling, WRG-Jahresverlauf und Zonenzuordnung prüfen.
3. `model.html` und `envelope.html`: Energie-/Ökobilanzgrafiken, Blattspalte, Verweise und RC-Fußnote prüfen.
4. `variants.html` und `declaration.html`: verschobene Erläuterungen und Linkziele prüfen.
5. Auch auf schmalem Bildschirm: keine abgeschnittenen Texte, Tabellen horizontal lesbar, Bilder vergrößerbar.

Die Preview im Chat zur Abnahme vorlegen. Hypothesis-Kommentare bleiben erhalten; erst im vereinbarten Freigabe-/Veröffentlichungsablauf löschen. Der Preset-Kommentar ist durch eine Skizze noch nicht erledigt.
